# Use Case 03: Medical Record Summarization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai/usecases/notebooks/03-medical-record-summary.ipynb)

**Companion notebook for**: `03-medical-record-summary.html`

## What This Notebook Covers
- Build a complete medical record summarization pipeline using LLMs
- De-identify clinical text (HIPAA Safe Harbor method)
- Extract medical entities: medications, diagnoses, allergies, labs, vitals
- Generate standardized SOAP notes from free-text clinical notes
- Check for dangerous drug-drug interactions
- Generate chronological patient timelines
- Produce I-PASS shift handoff summaries

## Important
- All patient data in this notebook is **100% synthetic** — no real patient information is used
- This is an educational demonstration, NOT a clinical tool
- Real clinical deployments require HIPAA compliance, clinician oversight, and regulatory approval

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q openai pandas

In [ ]:
import os
import re
import json
import textwrap
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional
from itertools import combinations

import pandas as pd
from openai import OpenAI

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
client = OpenAI()
print("All imports successful. OpenAI API key found.")

---
## Section 1: Synthetic Clinical Data

We create realistic (but entirely fictional) clinical notes representing different encounter types:
- Emergency department visit
- Discharge summary
- Progress note
- Outpatient follow-up

These notes use real clinical terminology, abbreviations, and formatting patterns that mirror actual medical documentation.

In [ ]:
# --- Synthetic Clinical Notes ---
# All patient data is 100% fictional. No real PHI.

CLINICAL_NOTES = [
    {
        "note_id": "N001",
        "encounter_type": "Emergency Department",
        "date": "2025-01-15",
        "time": "14:32",
        "author": "Dr. Sarah Chen",
        "patient": {
            "name": "Mr. Robert Johnson",
            "mrn": "MRN# 7834921",
            "dob": "03/15/1957",
            "phone": "(555) 867-5309",
            "ssn": "123-45-6789",
        },
        "text": (
            "EMERGENCY DEPARTMENT NOTE\n"
            "Date: 01/15/2025  Time: 14:32\n"
            "Patient: Mr. Robert Johnson  MRN# 7834921  DOB: 03/15/1957\n"
            "Provider: Dr. Sarah Chen\n\n"
            "CHIEF COMPLAINT: Chest pain, shortness of breath\n\n"
            "HPI: 67 y/o male presents to ED via EMS with acute onset substernal "
            "chest pain radiating to left arm x 2 hours. Associated with diaphoresis, "
            "nausea, and SOB. Pain rated 8/10, not relieved by rest or sublingual NTG "
            "administered by EMS. Hx of HTN, DM2, hyperlipidemia, and 40-pack-year "
            "smoking history (quit 5 years ago). Takes metformin 1000mg BID, "
            "atorvastatin 40mg daily, lisinopril 20mg daily.\n"
            "Allergies: PCN (rash), sulfa (hives).\n\n"
            "VITALS: BP 158/92, HR 104, RR 22, SpO2 94% on RA, Temp 98.6F\n\n"
            "PHYSICAL EXAM:\n"
            "General: Alert, diaphoretic, moderate distress\n"
            "HEENT: PERRL, no JVD\n"
            "Cardiovascular: Tachycardic, regular rhythm, no murmurs, rubs, or gallops\n"
            "Respiratory: Bilateral crackles at bases, no wheezing\n"
            "Abdomen: Soft, non-tender, non-distended, +BS\n"
            "Extremities: No edema, 2+ pulses bilaterally\n"
            "Neurological: A&O x3, no focal deficits\n\n"
            "LABS:\n"
            "Troponin I: 2.4 ng/mL (H)\n"
            "BNP: 1240 pg/mL (H)\n"
            "Cr: 1.6 mg/dL (H)\n"
            "K+: 4.8 mEq/L\n"
            "Glucose: 234 mg/dL (H)\n"
            "WBC: 11.2 K/uL\n"
            "Hgb: 13.1 g/dL\n"
            "Plt: 198 K/uL\n"
            "Na: 138 mEq/L\n"
            "BUN: 28 mg/dL (H)\n\n"
            "ECG: ST elevation in leads II, III, aVF consistent with inferior STEMI. "
            "Sinus tachycardia at 104 bpm.\n\n"
            "CXR: Mild pulmonary vascular congestion. No consolidation or effusion.\n\n"
            "ASSESSMENT:\n"
            "1. Acute inferior STEMI\n"
            "2. Acute decompensated heart failure\n"
            "3. Type 2 diabetes mellitus, uncontrolled (glucose 234)\n"
            "4. Acute kidney injury (Cr 1.6, baseline 1.0)\n\n"
            "PLAN:\n"
            "- Emergent cardiology consult for cardiac catheterization\n"
            "- Aspirin 325mg PO x1, clopidogrel 600mg loading dose\n"
            "- Heparin drip per ACS protocol\n"
            "- IV furosemide 40mg for pulmonary congestion\n"
            "- Hold metformin (AKI + pending contrast)\n"
            "- Continuous telemetry monitoring\n"
            "- Admit to CCU\n"
            "- Contact: (555) 867-5309 for family notification\n"
        ),
    },
    {
        "note_id": "N002",
        "encounter_type": "Discharge Summary",
        "date": "2025-01-20",
        "time": "10:15",
        "author": "Dr. James Williams",
        "patient": {
            "name": "Mr. Robert Johnson",
            "mrn": "MRN# 7834921",
            "dob": "03/15/1957",
            "phone": "(555) 867-5309",
            "ssn": "123-45-6789",
        },
        "text": (
            "DISCHARGE SUMMARY\n"
            "Patient: Mr. Robert Johnson  MRN# 7834921  DOB: 03/15/1957\n"
            "Admission: 01/15/2025   Discharge: 01/20/2025\n"
            "Attending: Dr. James Williams, Cardiology\n\n"
            "ADMITTING DIAGNOSIS: Acute inferior STEMI (ICD-10: I21.19)\n\n"
            "DISCHARGE DIAGNOSES:\n"
            "1. Acute inferior STEMI, s/p PCI with DES to RCA (I21.19)\n"
            "2. Acute decompensated heart failure, now compensated (I50.21)\n"
            "3. Type 2 diabetes mellitus, uncontrolled (E11.65)\n"
            "4. Hypertension (I10)\n"
            "5. Acute kidney injury, stage 1, resolved (N17.9)\n"
            "6. Hyperlipidemia (E78.5)\n\n"
            "HOSPITAL COURSE:\n"
            "Patient underwent emergent cardiac catheterization on 01/15/2025 "
            "revealing 95% occlusion of the RCA. Successful PCI with drug-eluting "
            "stent placement. Post-procedure course complicated by:\n"
            "- Troponin peaked at 18.2 ng/mL on HD#2, then trended down\n"
            "- AKI with Cr peaking at 2.1 mg/dL on HD#2 (contrast + hemodynamic), "
            "resolved to 1.2 mg/dL at discharge with IV hydration\n"
            "- CHF managed with IV furosemide diuresis, converted to oral on HD#3. "
            "Repeat CXR showed improved congestion.\n"
            "- Echo on HD#2: EF 35%, inferior wall hypokinesis, mild MR\n"
            "- A1C 9.2% - endocrine consulted, started on insulin glargine\n"
            "- Tolerated cardiac diet, ambulating with PT on HD#4\n"
            "- Remained on telemetry without significant arrhythmias\n\n"
            "DISCHARGE VITALS: BP 124/78, HR 72, RR 16, SpO2 97% on RA\n\n"
            "DISCHARGE LABS:\n"
            "Troponin I: 1.8 ng/mL (trending down)\n"
            "Cr: 1.2 mg/dL (improved)\n"
            "K+: 4.2 mEq/L\n"
            "BNP: 580 pg/mL (improved from 1240)\n"
            "Glucose: 156 mg/dL\n"
            "A1C: 9.2%\n\n"
            "DISCHARGE MEDICATIONS:\n"
            "1. Aspirin 81mg PO daily (DO NOT STOP - stent)\n"
            "2. Clopidogrel 75mg PO daily x 12 months (DO NOT STOP - stent)\n"
            "3. Atorvastatin 80mg PO daily (increased from 40mg)\n"
            "4. Metoprolol succinate 50mg PO daily (NEW)\n"
            "5. Lisinopril 10mg PO daily (decreased from 20mg due to AKI)\n"
            "6. Metformin 1000mg PO BID (HOLD if Cr > 1.5)\n"
            "7. Insulin glargine 20 units SubQ at bedtime (NEW)\n"
            "8. Furosemide 40mg PO daily (NEW)\n"
            "9. KCl 20mEq PO daily (NEW - with furosemide)\n\n"
            "ALLERGIES: Penicillin (rash), Sulfonamides (hives)\n\n"
            "FOLLOW-UP:\n"
            "- Cardiology (Dr. Williams): 1 week - call (555) 123-4567\n"
            "- PCP (Dr. Martinez): 2 weeks\n"
            "- Endocrinology: 1 month for DM management\n"
            "- Cardiac rehab referral placed\n"
            "- Return to ED if chest pain, SOB, or weight gain > 3 lbs/day\n"
        ),
    },
    {
        "note_id": "N003",
        "encounter_type": "Progress Note",
        "date": "2025-01-17",
        "time": "07:45",
        "author": "Dr. James Williams",
        "patient": {
            "name": "Mr. Robert Johnson",
            "mrn": "MRN# 7834921",
            "dob": "03/15/1957",
            "phone": "(555) 867-5309",
            "ssn": "123-45-6789",
        },
        "text": (
            "PROGRESS NOTE - Hospital Day #3\n"
            "Date: 01/17/2025  Time: 07:45\n"
            "Patient: Mr. Robert Johnson  MRN# 7834921\n"
            "Service: Cardiology  Attending: Dr. James Williams\n\n"
            "S: Patient reports feeling 'much better today'. Chest pain resolved. "
            "Mild SOB with exertion (walking to bathroom) but improved from yesterday. "
            "No orthopnea. Appetite improving, tolerated cardiac diet. Denies dizziness, "
            "palpitations, or lower extremity swelling. Anxious about going home and "
            "managing new medications.\n\n"
            "O:\n"
            "Vitals: BP 132/80, HR 78, RR 18, SpO2 96% on RA, Temp 98.4F\n"
            "I/O: Intake 1800mL, Output 2400mL (net -600mL)\n"
            "Weight: 92.1 kg (down 1.8 kg from admission)\n\n"
            "Exam: NAD. JVP ~8cm. Lungs: faint crackles at L base only (improved). "
            "Heart: RRR, no murmurs. Abdomen benign. Trace pedal edema (improved).\n\n"
            "Labs (AM draw):\n"
            "Troponin I: 6.4 ng/mL (down from 18.2 peak)\n"
            "Cr: 1.8 mg/dL (up from 1.6, concerning)\n"
            "K+: 5.1 mEq/L (H)\n"
            "BNP: 890 pg/mL (down from 1240)\n"
            "Glucose: 198 mg/dL (fasting)\n\n"
            "Echo (01/16): EF 35%, inferior wall hypokinesis, mild MR, no pericardial "
            "effusion. LA mildly dilated.\n\n"
            "A:\n"
            "1. STEMI s/p PCI to RCA - clinically improving, troponin trending down\n"
            "2. CHF (EF 35%) - improving with diuresis, still volume overloaded\n"
            "3. AKI - Cr worsening to 1.8 despite hydration. Likely cardiorenal. "
            "Holding ACE-I and adjusting diuretic dose.\n"
            "4. Hyperkalemia (K+ 5.1) - likely AKI-related. Concerning given ACE-I use.\n"
            "5. DM2 uncontrolled - fasting glucose 198. Endocrine to see today.\n\n"
            "P:\n"
            "- Continue dual antiplatelet (aspirin + clopidogrel) - critical for stent\n"
            "- HOLD lisinopril given AKI + hyperkalemia\n"
            "- Decrease furosemide to 20mg IV BID (balance diuresis vs AKI)\n"
            "- Kayexalate 15g PO x1 for K+ 5.1\n"
            "- Recheck BMP in AM\n"
            "- Endocrine consult for insulin dosing\n"
            "- PT evaluation for cardiac rehab readiness\n"
            "- Continue telemetry\n"
            "- Goal: discharge HD#5-6 if Cr improving and K+ normalizes\n"
        ),
    },
    {
        "note_id": "N004",
        "encounter_type": "Outpatient Follow-up",
        "date": "2025-01-27",
        "time": "09:30",
        "author": "Dr. James Williams",
        "patient": {
            "name": "Mr. Robert Johnson",
            "mrn": "MRN# 7834921",
            "dob": "03/15/1957",
            "phone": "(555) 867-5309",
            "ssn": "123-45-6789",
        },
        "text": (
            "OUTPATIENT CARDIOLOGY FOLLOW-UP\n"
            "Date: 01/27/2025  Time: 09:30\n"
            "Patient: Mr. Robert Johnson  MRN# 7834921  DOB: 03/15/1957\n"
            "Provider: Dr. James Williams, Cardiology\n\n"
            "1-WEEK POST-DISCHARGE FOLLOW-UP (s/p inferior STEMI, PCI to RCA)\n\n"
            "S: Patient reports doing well overall. No recurrence of chest pain. "
            "Mild exertional dyspnea when climbing stairs (1 flight), improved from "
            "hospital. No orthopnea, PND, or LE swelling. Taking all medications as "
            "prescribed. Checking blood sugars - fasting 140-170 range. Wife helping "
            "with insulin injections. No hypoglycemic episodes. Diet compliant - "
            "low sodium, cardiac diet. Walking 10 minutes daily per PT instructions.\n\n"
            "O:\n"
            "Vitals: BP 128/76, HR 68, RR 16, SpO2 98% on RA\n"
            "Weight: 90.3 kg (down 1.8 kg from discharge)\n\n"
            "Exam: Well-appearing, NAD. JVP normal. Lungs CTA bilaterally. "
            "Heart: RRR, S1/S2 normal, no murmurs or gallops. No peripheral edema. "
            "Cath site (R femoral): clean, no hematoma, no bruit.\n\n"
            "Labs (drawn 01/25):\n"
            "Cr: 1.1 mg/dL (normalized!)\n"
            "K+: 4.4 mEq/L\n"
            "BNP: 320 pg/mL (improving)\n"
            "Glucose: 152 mg/dL\n"
            "LDL: 68 mg/dL (at goal on atorvastatin 80mg)\n\n"
            "A:\n"
            "1. CAD s/p STEMI, s/p PCI w/DES to RCA - stable, no recurrent ischemia\n"
            "2. Systolic HF (EF 35%) - NYHA class II, euvolemic on exam\n"
            "3. AKI - RESOLVED (Cr 1.1)\n"
            "4. DM2 - improved but not at goal (fasting 140-170)\n"
            "5. HTN - well controlled on current regimen\n\n"
            "P:\n"
            "- Continue DAPT (aspirin 81mg + clopidogrel 75mg) - DO NOT STOP x 12mo\n"
            "- Continue atorvastatin 80mg - LDL at goal\n"
            "- Continue metoprolol succinate 50mg daily\n"
            "- RESTART lisinopril 10mg daily (Cr normalized, K+ normal)\n"
            "- Continue furosemide 40mg daily\n"
            "- Continue KCl 20mEq daily\n"
            "- Continue metformin 1000mg BID (Cr < 1.5, safe to continue)\n"
            "- Increase insulin glargine to 24 units at bedtime (from 20)\n"
            "- Cardiac rehab: cleared to start, referral confirmed\n"
            "- Repeat echo in 3 months to reassess EF\n"
            "- Follow up in 1 month or sooner if symptoms\n"
            "- Continue low-sodium cardiac diet\n"
        ),
    },
]

print(f"Loaded {len(CLINICAL_NOTES)} synthetic clinical notes:")
for note in CLINICAL_NOTES:
    print(f"  [{note['note_id']}] {note['date']} - {note['encounter_type']} ({len(note['text'])} chars)")

---
## Section 2: De-identification (HIPAA Safe Harbor)

Before any clinical text is sent to an LLM API, all Protected Health Information (PHI) must be removed. HIPAA defines 18 categories of identifiers. We implement regex-based de-identification for common PHI types.

In production, use a validated de-identification tool (Philter, NLM Scrubber, etc.) rather than custom regex.

In [ ]:
# --- HIPAA Safe Harbor De-identification ---

PHI_PATTERNS: Dict[str, str] = {
    # Patient names (Dr./Mr./Mrs./Ms. + capitalized words)
    "NAME": r"\b(?:Dr\.|Mr\.|Mrs\.|Ms\.)\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+",
    # Dates in various formats
    "DATE": r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b",
    # Phone numbers
    "PHONE": r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}",
    # SSN
    "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
    # Medical Record Numbers
    "MRN": r"MRN[:#\s]*\d{6,10}",
    # Email addresses
    "EMAIL": r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
}


def deidentify(text: str) -> Tuple[str, List[dict]]:
    """
    Remove PHI from clinical text using HIPAA Safe Harbor method.
    Returns (cleaned_text, list_of_redactions_for_audit_log).
    """
    redactions = []
    cleaned = text

    # Process SSN first (before DATE, since SSN pattern overlaps)
    ordered_types = ["SSN", "MRN", "PHONE", "EMAIL", "NAME", "DATE"]

    for phi_type in ordered_types:
        pattern = PHI_PATTERNS[phi_type]
        for match in re.finditer(pattern, cleaned):
            redactions.append({
                "type": phi_type,
                "original": match.group(),
                "start": match.start(),
                "end": match.end(),
            })
        cleaned = re.sub(pattern, f"[{phi_type}]", cleaned)

    return cleaned, redactions


# Demo: de-identify the first clinical note
sample_note = CLINICAL_NOTES[0]["text"]
cleaned_note, redactions = deidentify(sample_note)

print("=" * 60)
print("DE-IDENTIFICATION RESULTS")
print("=" * 60)
print(f"\nRedactions found: {len(redactions)}")
for r in redactions:
    print(f"  [{r['type']}] '{r['original']}'")

print(f"\n--- Cleaned Note (first 500 chars) ---")
print(cleaned_note[:500])

In [ ]:
# De-identify ALL notes and store for downstream processing

deidentified_notes = []
total_redactions = 0

for note in CLINICAL_NOTES:
    cleaned, redactions = deidentify(note["text"])
    deidentified_notes.append({
        "note_id": note["note_id"],
        "encounter_type": note["encounter_type"],
        "date": note["date"],
        "cleaned_text": cleaned,
        "redaction_count": len(redactions),
    })
    total_redactions += len(redactions)

print(f"De-identified {len(deidentified_notes)} notes")
print(f"Total PHI redactions: {total_redactions}")
for dn in deidentified_notes:
    print(f"  [{dn['note_id']}] {dn['encounter_type']}: {dn['redaction_count']} redactions")

---
## Section 3: Medical Entity Extraction

We use a **hybrid approach**:
1. **Regex patterns** capture well-formatted entities (vitals, lab values, structured medication lists)
2. **LLM extraction** catches entities embedded in narrative text that regex misses

The two layers are merged, with regex providing high-precision anchors and the LLM providing high-recall coverage.

In [ ]:
# --- Regex-Based Entity Extraction ---

@dataclass
class Medication:
    name: str
    dose: str
    route: str
    frequency: str
    status: str = "active"  # active, discontinued, hold, new

@dataclass
class LabResult:
    test_name: str
    value: str
    unit: str
    flag: str = "normal"  # normal, high, low, critical

@dataclass
class VitalSigns:
    bp: str = ""
    hr: str = ""
    rr: str = ""
    spo2: str = ""
    temp: str = ""


# --- Vital Signs Extraction ---
def extract_vitals(text: str) -> VitalSigns:
    """Extract vital signs using regex patterns."""
    vitals = VitalSigns()
    bp_match = re.search(r"BP\s*:?\s*(\d{2,3}/\d{2,3})", text)
    hr_match = re.search(r"HR\s*:?\s*(\d{2,3})", text)
    rr_match = re.search(r"RR\s*:?\s*(\d{1,2})", text)
    spo2_match = re.search(r"SpO2\s*:?\s*(\d{2,3})%?", text)
    temp_match = re.search(r"Temp\s*:?\s*(\d{2,3}(?:\.\d)?)\s*F?", text)

    if bp_match: vitals.bp = bp_match.group(1)
    if hr_match: vitals.hr = hr_match.group(1)
    if rr_match: vitals.rr = rr_match.group(1)
    if spo2_match: vitals.spo2 = spo2_match.group(1)
    if temp_match: vitals.temp = temp_match.group(1)
    return vitals


# --- Lab Results Extraction ---
LAB_PATTERN = re.compile(
    r"(Troponin\s*I?|BNP|Cr|K\+?|Glucose|WBC|Hgb|Plt|Na|BUN|A1C|LDL|AST|ALT)"
    r"\s*:?\s*"
    r"(\d+(?:\.\d+)?)"
    r"\s*"
    r"(ng/mL|pg/mL|mg/dL|mmol/L|mEq/L|%|K/uL|g/dL)?",
    re.IGNORECASE
)

# Normal ranges for flagging
NORMAL_RANGES = {
    "troponin": (0, 0.04, "ng/mL"),
    "bnp": (0, 100, "pg/mL"),
    "cr": (0.6, 1.2, "mg/dL"),
    "k": (3.5, 5.0, "mEq/L"),
    "glucose": (70, 100, "mg/dL"),
    "wbc": (4.5, 11.0, "K/uL"),
    "hgb": (12.0, 17.5, "g/dL"),
    "plt": (150, 400, "K/uL"),
    "na": (136, 145, "mEq/L"),
    "bun": (7, 20, "mg/dL"),
    "a1c": (4.0, 5.7, "%"),
    "ldl": (0, 100, "mg/dL"),
}

def extract_labs(text: str) -> List[LabResult]:
    """Extract lab results and flag abnormal values."""
    labs = []
    for match in LAB_PATTERN.finditer(text):
        name = match.group(1).strip()
        value = match.group(2)
        unit = match.group(3) or ""

        # Determine flag
        flag = "normal"
        name_key = name.lower().replace(" ", "").replace("+", "").replace("i", "")
        # Map common variations
        name_map = {"troponin": "troponin", "k": "k", "cr": "cr"}
        for key in NORMAL_RANGES:
            if key in name_key or name_key in key:
                low, high, _ = NORMAL_RANGES[key]
                val = float(value)
                if val > high:
                    flag = "critical" if val > high * 2 else "high"
                elif val < low:
                    flag = "low"
                break

        labs.append(LabResult(
            test_name=name,
            value=value,
            unit=unit,
            flag=flag,
        ))
    return labs


# --- Demo: Extract from ER note ---
er_note = deidentified_notes[0]["cleaned_text"]

vitals = extract_vitals(er_note)
labs = extract_labs(er_note)

print("VITAL SIGNS (Regex Extraction):")
print(f"  BP: {vitals.bp}")
print(f"  HR: {vitals.hr}")
print(f"  RR: {vitals.rr}")
print(f"  SpO2: {vitals.spo2}%")
print(f"  Temp: {vitals.temp}F")

print(f"\nLAB RESULTS (Regex Extraction): {len(labs)} found")
for lab in labs:
    flag_marker = f" *** {lab.flag.upper()} ***" if lab.flag != "normal" else ""
    print(f"  {lab.test_name}: {lab.value} {lab.unit}{flag_marker}")

In [ ]:
# --- LLM-Based Entity Extraction ---
# Catches entities in narrative text that regex misses

ENTITY_EXTRACTION_PROMPT = """You are a clinical NLP system. Extract ALL medical
entities from the following de-identified clinical note.

Return valid JSON with this exact schema:
{
  "medications": [
    {"name": "...", "dose": "...", "route": "...", "frequency": "...",
     "status": "active|new|hold|discontinued", "confidence": 0.0}
  ],
  "diagnoses": [
    {"description": "...", "icd10_code": "...",
     "status": "active|resolved|chronic", "confidence": 0.0}
  ],
  "allergies": [
    {"substance": "...", "reaction": "...", "severity": "mild|moderate|severe"}
  ],
  "procedures": [
    {"name": "...", "date": "...", "details": "..."}
  ]
}

RULES:
- confidence: 0.95+ for explicitly stated, 0.6-0.8 for inferred, <0.5 for uncertain
- Include ICD-10 codes when identifiable
- Mark medication status: "new" if started during encounter, "hold" if held,
  "discontinued" if stopped, "active" if continuing
- Do NOT hallucinate entities not present in the text"""


def extract_entities_llm(clinical_note: str) -> dict:
    """Use LLM to extract structured medical entities from clinical text."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ENTITY_EXTRACTION_PROMPT},
            {"role": "user", "content": clinical_note},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=2048,
    )
    return json.loads(response.choices[0].message.content)


# Extract from the ER note
er_entities = extract_entities_llm(deidentified_notes[0]["cleaned_text"])

print("LLM ENTITY EXTRACTION - ER Note")
print("=" * 50)

print(f"\nMedications ({len(er_entities.get('medications', []))})")
for med in er_entities.get("medications", []):
    conf = med.get('confidence', 'N/A')
    print(f"  [{med['status'].upper():>12}] {med['name']} {med['dose']} "
          f"{med['route']} {med['frequency']} (conf: {conf})")

print(f"\nDiagnoses ({len(er_entities.get('diagnoses', []))})")
for dx in er_entities.get("diagnoses", []):
    conf = dx.get('confidence', 'N/A')
    print(f"  [{dx['status'].upper():>8}] {dx['description']} "
          f"({dx.get('icd10_code', 'N/A')}) conf: {conf}")

print(f"\nAllergies ({len(er_entities.get('allergies', []))})")
for allergy in er_entities.get("allergies", []):
    print(f"  {allergy['substance']}: {allergy['reaction']} ({allergy['severity']})")

print(f"\nProcedures ({len(er_entities.get('procedures', []))})")
for proc in er_entities.get("procedures", []):
    print(f"  {proc['name']} ({proc.get('date', 'N/A')})")

In [ ]:
# --- Extract entities from ALL notes ---

all_entities = {}
for note in deidentified_notes:
    print(f"Extracting entities from [{note['note_id']}] {note['encounter_type']}...")
    entities = extract_entities_llm(note["cleaned_text"])
    all_entities[note["note_id"]] = entities
    n_meds = len(entities.get("medications", []))
    n_dx = len(entities.get("diagnoses", []))
    n_allergy = len(entities.get("allergies", []))
    print(f"  -> {n_meds} medications, {n_dx} diagnoses, {n_allergy} allergies")

print(f"\nEntity extraction complete for {len(all_entities)} notes.")

---
## Section 4: SOAP Note Generation

SOAP (Subjective, Objective, Assessment, Plan) is the universal standard for clinical documentation. We use the LLM to convert free-text notes into structured SOAP format with JSON output for downstream processing.

In [ ]:
SOAP_PROMPT = """You are an expert clinical documentation assistant.
Convert the following clinical note into a structured SOAP note.

Return valid JSON with this schema:
{
  "subjective": {
    "chief_complaint": "...",
    "hpi": "...",
    "past_medical_history": ["..."],
    "medications_reported": ["..."],
    "allergies": [{"substance": "...", "reaction": "..."}],
    "social_history": "...",
    "review_of_systems": "..."
  },
  "objective": {
    "vitals": {"bp": "...", "hr": "...", "rr": "...", "spo2": "...", "temp": "..."},
    "physical_exam": {
      "general": "...",
      "cardiovascular": "...",
      "respiratory": "...",
      "abdomen": "...",
      "extremities": "...",
      "neurological": "..."
    },
    "labs": [{"test": "...", "value": "...", "unit": "...", "flag": "normal|high|low|critical"}],
    "imaging": "...",
    "ecg": "..."
  },
  "assessment": {
    "primary_diagnosis": {"description": "...", "icd10": "..."},
    "secondary_diagnoses": [{"description": "...", "icd10": "..."}],
    "clinical_reasoning": "..."
  },
  "plan": {
    "medications": [{"action": "start|continue|hold|increase|decrease|stop",
                     "drug": "...", "dose": "...", "route": "...",
                     "frequency": "...", "rationale": "..."}],
    "procedures": ["..."],
    "consults": ["..."],
    "follow_up": "...",
    "disposition": "..."
  }
}

RULES:
- Place information in the correct SOAP section
- Include ALL medications with exact dosages
- Flag abnormal lab values with flag field
- Do NOT infer information not in the source note
- If a section has no data, use "Not documented"
- Preserve clinical precision (exact values, no rounding)"""


def generate_soap_note(clinical_note: str) -> dict:
    """Convert free-text clinical note to structured SOAP format."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SOAP_PROMPT},
            {"role": "user", "content": f"Clinical Note:\n{clinical_note}"},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=4096,
    )
    return json.loads(response.choices[0].message.content)


# Generate SOAP note from the ER encounter
er_soap = generate_soap_note(deidentified_notes[0]["cleaned_text"])

print("SOAP NOTE - Emergency Department Encounter")
print("=" * 55)

# Subjective
print("\n--- SUBJECTIVE ---")
print(f"Chief Complaint: {er_soap['subjective']['chief_complaint']}")
print(f"HPI: {er_soap['subjective']['hpi'][:200]}...")
print(f"PMH: {', '.join(er_soap['subjective'].get('past_medical_history', []))}")
print(f"Allergies:")
for a in er_soap['subjective'].get('allergies', []):
    print(f"  - {a['substance']}: {a['reaction']}")

# Objective
print("\n--- OBJECTIVE ---")
v = er_soap['objective']['vitals']
print(f"Vitals: BP {v['bp']}, HR {v['hr']}, RR {v['rr']}, SpO2 {v['spo2']}, Temp {v['temp']}")
print(f"Labs:")
for lab in er_soap['objective'].get('labs', [])[:5]:
    flag = f" [{lab['flag'].upper()}]" if lab['flag'] != 'normal' else ""
    print(f"  {lab['test']}: {lab['value']} {lab.get('unit', '')}{flag}")
if len(er_soap['objective'].get('labs', [])) > 5:
    print(f"  ... and {len(er_soap['objective']['labs']) - 5} more")

# Assessment
print("\n--- ASSESSMENT ---")
primary = er_soap['assessment']['primary_diagnosis']
print(f"Primary: {primary['description']} ({primary.get('icd10', 'N/A')})")
for dx in er_soap['assessment'].get('secondary_diagnoses', []):
    print(f"  Secondary: {dx['description']} ({dx.get('icd10', 'N/A')})")

# Plan
print("\n--- PLAN ---")
for med in er_soap['plan'].get('medications', []):
    print(f"  [{med['action'].upper()}] {med['drug']} {med['dose']} {med.get('route', '')} "
          f"{med.get('frequency', '')}")

---
## Section 5: Drug Interaction Checking

After extracting all medications, we cross-reference every pair against a drug interaction database. In production, this would use FDA data, DrugBank, or Lexicomp. Here we use a curated synthetic database of clinically significant interactions.

In [ ]:
# --- Drug Interaction Database ---
# Synthetic subset of clinically significant interactions

DRUG_INTERACTIONS = {
    ("warfarin", "aspirin"): {
        "severity": "HIGH",
        "effect": "Increased bleeding risk from combined anticoagulant + antiplatelet effect.",
        "recommendation": "Monitor INR closely. Consider PPI for GI protection.",
    },
    ("metformin", "contrast dye"): {
        "severity": "HIGH",
        "effect": "Risk of lactic acidosis. IV contrast can cause AKI, impairing metformin clearance.",
        "recommendation": "Hold metformin 48h before and after contrast. Check Cr before restarting.",
    },
    ("lisinopril", "potassium"): {
        "severity": "MODERATE",
        "effect": "ACE inhibitors reduce K+ excretion. Concurrent K+ supplementation increases hyperkalemia risk.",
        "recommendation": "Monitor serum K+ weekly. Target 3.5-5.0 mEq/L. Stop supplement if K+ > 5.0.",
    },
    ("lisinopril", "kcl"): {
        "severity": "MODERATE",
        "effect": "ACE inhibitor + potassium chloride increases hyperkalemia risk.",
        "recommendation": "Monitor serum potassium. Reduce or stop KCl if K+ > 5.0.",
    },
    ("metoprolol", "verapamil"): {
        "severity": "HIGH",
        "effect": "Combined beta-blocker + CCB can cause severe bradycardia, heart block, hypotension.",
        "recommendation": "Avoid combination. If necessary, monitor ECG and BP closely.",
    },
    ("clopidogrel", "omeprazole"): {
        "severity": "MODERATE",
        "effect": "Omeprazole inhibits CYP2C19, reducing clopidogrel activation. Reduced antiplatelet effect.",
        "recommendation": "Use pantoprazole instead. Critical for patients with recent stent.",
    },
    ("furosemide", "lisinopril"): {
        "severity": "MODERATE",
        "effect": "Loop diuretic-induced hypovolemia potentiates ACE inhibitor hypotension. AKI risk.",
        "recommendation": "Monitor BP, Cr, and electrolytes. Consider dose reduction of ACE-I.",
    },
    ("insulin", "metformin"): {
        "severity": "LOW",
        "effect": "Additive hypoglycemic effect. Combination is therapeutically intended but increases hypo risk.",
        "recommendation": "Educate patient on hypoglycemia signs. Monitor blood glucose regularly.",
    },
    ("aspirin", "clopidogrel"): {
        "severity": "LOW",
        "effect": "Dual antiplatelet therapy (DAPT) increases bleeding risk. Therapeutically intended post-stent.",
        "recommendation": "Expected combination post-PCI. Monitor for bleeding signs. GI protection with PPI.",
    },
    ("atorvastatin", "grapefruit"): {
        "severity": "LOW",
        "effect": "Grapefruit inhibits CYP3A4, increasing statin levels and myopathy risk.",
        "recommendation": "Avoid grapefruit consumption. Monitor for muscle pain.",
    },
}


def normalize_drug_name(name: str) -> str:
    """Normalize drug names for interaction lookup."""
    name = name.lower().strip()
    # Map brand names and variations to generic names
    brand_to_generic = {
        "lopressor": "metoprolol",
        "toprol": "metoprolol",
        "metoprolol succinate": "metoprolol",
        "metoprolol tartrate": "metoprolol",
        "glucophage": "metformin",
        "lipitor": "atorvastatin",
        "plavix": "clopidogrel",
        "prinivil": "lisinopril",
        "zestril": "lisinopril",
        "lasix": "furosemide",
        "coumadin": "warfarin",
        "insulin glargine": "insulin",
        "insulin lispro": "insulin",
        "potassium chloride": "kcl",
    }
    return brand_to_generic.get(name, name)


def check_interactions(medication_names: List[str]) -> List[dict]:
    """Check all medication pairs for known interactions."""
    normalized = [normalize_drug_name(m) for m in medication_names]
    interactions_found = []

    for i, j in combinations(range(len(normalized)), 2):
        pair = (normalized[i], normalized[j])
        pair_rev = (normalized[j], normalized[i])

        interaction = DRUG_INTERACTIONS.get(pair) or DRUG_INTERACTIONS.get(pair_rev)
        if interaction:
            interactions_found.append({
                "drug_a": medication_names[i],
                "drug_b": medication_names[j],
                **interaction,
            })

    # Sort by severity: HIGH > MODERATE > LOW
    severity_order = {"HIGH": 0, "MODERATE": 1, "LOW": 2}
    return sorted(interactions_found, key=lambda x: severity_order[x["severity"]])


# --- Check interactions for discharge medication list ---
discharge_meds = [
    "Aspirin", "Clopidogrel", "Atorvastatin",
    "Metoprolol succinate", "Lisinopril", "Metformin",
    "Insulin glargine", "Furosemide", "KCl",
]

interactions = check_interactions(discharge_meds)

print("DRUG INTERACTION CHECK")
print(f"Medications checked: {len(discharge_meds)}")
print(f"Pairs evaluated: {len(list(combinations(discharge_meds, 2)))}")
print(f"Interactions found: {len(interactions)}")
print("=" * 60)

for ix in interactions:
    severity_icon = {"HIGH": "!!!", "MODERATE": " ! ", "LOW": " . "}[ix["severity"]]
    print(f"\n[{severity_icon}] {ix['severity']} SEVERITY")
    print(f"    {ix['drug_a']}  <-->  {ix['drug_b']}")
    print(f"    Effect: {ix['effect']}")
    print(f"    Action: {ix['recommendation']}")

---
## Section 6: Patient Timeline Generation

We compile all encounters into a chronological timeline, highlighting key events: admissions, procedures, lab trends, medication changes, and discharge.

In [ ]:
TIMELINE_PROMPT = """Analyze these clinical notes for the SAME patient and generate
a chronological timeline. Return valid JSON:

{
  "patient_summary": "One-line patient summary",
  "timeline": [
    {
      "date": "YYYY-MM-DD",
      "time": "HH:MM or null",
      "event_type": "admission|procedure|lab_result|medication_change|diagnosis|imaging|discharge|follow_up",
      "description": "Brief clinical description",
      "significance": "routine|notable|critical"
    }
  ],
  "trajectory": "improving|stable|declining|mixed",
  "key_turning_points": ["Brief description of each turning point"],
  "active_problems": ["Current active medical problems"],
  "resolved_problems": ["Problems that have resolved"]
}

RULES:
- Order events chronologically
- Mark critical events (new diagnoses, procedures, critical labs) as significance=critical
- Track medication changes across encounters
- Identify the overall trajectory and key turning points
- Be concise but clinically precise"""


def generate_timeline(notes: List[dict]) -> dict:
    """Generate a chronological patient timeline from multiple clinical notes."""
    # Combine all notes with clear separators
    combined = "\n\n" + "=" * 40 + "\n\n"
    combined = combined.join([
        f"[{n['note_id']}] {n['encounter_type']} - {n['date']}\n{n['cleaned_text']}"
        for n in notes
    ])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": TIMELINE_PROMPT},
            {"role": "user", "content": combined},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=4096,
    )
    return json.loads(response.choices[0].message.content)


# Generate timeline from all notes
timeline = generate_timeline(deidentified_notes)

print("PATIENT TIMELINE")
print("=" * 60)
print(f"Summary: {timeline.get('patient_summary', 'N/A')}")
print(f"Overall Trajectory: {timeline.get('trajectory', 'N/A')}")

print(f"\nTimeline ({len(timeline.get('timeline', []))} events):")
for event in timeline.get("timeline", []):
    sig_marker = {"critical": "[!!!]", "notable": "[ ! ]", "routine": "[ . ]"}[
        event.get("significance", "routine")
    ]
    time_str = f" {event['time']}" if event.get('time') else ""
    print(f"  {event['date']}{time_str} {sig_marker} [{event['event_type']}]")
    print(f"    {event['description']}")

print(f"\nKey Turning Points:")
for tp in timeline.get("key_turning_points", []):
    print(f"  - {tp}")

print(f"\nActive Problems:")
for p in timeline.get("active_problems", []):
    print(f"  - {p}")

print(f"\nResolved Problems:")
for p in timeline.get("resolved_problems", []):
    print(f"  - {p}")

---
## Section 7: Handoff Summary (I-PASS Format)

The I-PASS framework is the gold standard for clinical handoffs:
- **I**llness severity (Stable / Watcher / Unstable)
- **P**atient summary (one-liner + key events)
- **A**ction list (pending tasks, meds due, labs to follow)
- **S**ituation awareness (what could go wrong, contingency plans)
- **S**ynthesis by receiver (read-back confirmation)

The LLM generates this from the SOAP notes, drug interactions, and timeline.

In [ ]:
HANDOFF_PROMPT = """You are a physician generating an I-PASS handoff summary.

Given the patient's SOAP note, drug interaction alerts, and timeline,
create a concise handoff that can be read in under 2 minutes.

Format:

I - ILLNESS SEVERITY: [Stable / Watcher / Unstable]

P - PATIENT SUMMARY:
[One-line summary]
[Key active problems]

A - ACTION LIST:
[Pending tasks, medications due, labs to check]

S - SITUATION AWARENESS:
[What could go wrong, contingency plans, critical alerts]

S - SYNTHESIS:
[Key points for receiving clinician, critical do-not-miss items]

RULES:
- Be concise and actionable
- Highlight any drug interaction alerts prominently
- Flag any critical lab trends
- Include specific vital sign and lab thresholds for escalation
- This is for a shift change, not a discharge summary"""


def generate_handoff(
    soap_note: dict,
    interactions: List[dict],
    timeline: dict,
) -> str:
    """Generate I-PASS handoff summary."""
    context = json.dumps({
        "soap_note": soap_note,
        "drug_interactions": interactions,
        "timeline_summary": {
            "trajectory": timeline.get("trajectory"),
            "active_problems": timeline.get("active_problems"),
            "key_turning_points": timeline.get("key_turning_points"),
            "recent_events": timeline.get("timeline", [])[-5:],
        },
    }, indent=2)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": HANDOFF_PROMPT},
            {"role": "user", "content": context},
        ],
        temperature=0.1,
        max_tokens=1500,
    )
    return response.choices[0].message.content


# Generate handoff using the ER SOAP note, interactions, and timeline
handoff = generate_handoff(er_soap, interactions, timeline)

print("I-PASS HANDOFF SUMMARY")
print("=" * 60)
print(handoff)

---
## Section 8: Full Pipeline — End-to-End

Now we wire the entire pipeline together: ingest a raw note, de-identify, extract entities, generate SOAP, check drug interactions, and produce a handoff summary.

In [ ]:
def run_full_pipeline(raw_note: str, note_id: str = "N000") -> dict:
    """
    Run the complete medical record summarization pipeline.

    Steps:
    1. De-identify (remove PHI)
    2. Extract medical entities (LLM)
    3. Extract vitals and labs (regex)
    4. Generate SOAP note (LLM)
    5. Check drug interactions
    6. Return structured results
    """
    results = {"note_id": note_id, "steps": {}}

    # Step 1: De-identification
    print(f"[1/5] De-identifying clinical note...")
    cleaned_text, redactions = deidentify(raw_note)
    results["steps"]["deidentification"] = {
        "redactions_count": len(redactions),
        "redaction_types": list(set(r["type"] for r in redactions)),
    }
    print(f"       Removed {len(redactions)} PHI elements")

    # Step 2: Entity extraction (LLM)
    print(f"[2/5] Extracting medical entities (LLM)...")
    entities = extract_entities_llm(cleaned_text)
    results["steps"]["entities"] = entities
    n_meds = len(entities.get("medications", []))
    n_dx = len(entities.get("diagnoses", []))
    print(f"       Found {n_meds} medications, {n_dx} diagnoses")

    # Step 3: Vitals and labs (regex)
    print(f"[3/5] Extracting vitals and labs (regex)...")
    vitals = extract_vitals(cleaned_text)
    labs = extract_labs(cleaned_text)
    results["steps"]["vitals"] = asdict(vitals)
    results["steps"]["labs"] = [asdict(lab) for lab in labs]
    abnormal = [l for l in labs if l.flag != "normal"]
    print(f"       {len(labs)} lab values, {len(abnormal)} abnormal")

    # Step 4: SOAP note generation
    print(f"[4/5] Generating SOAP note (LLM)...")
    soap = generate_soap_note(cleaned_text)
    results["steps"]["soap_note"] = soap
    print(f"       SOAP note generated")

    # Step 5: Drug interaction check
    print(f"[5/5] Checking drug interactions...")
    med_names = [m["name"] for m in entities.get("medications", [])]
    interactions = check_interactions(med_names)
    results["steps"]["interactions"] = interactions
    high_sev = [i for i in interactions if i["severity"] == "HIGH"]
    print(f"       {len(interactions)} interactions ({len(high_sev)} HIGH severity)")

    results["status"] = "complete"
    return results


# Run full pipeline on the discharge summary
print("FULL PIPELINE EXECUTION")
print("=" * 60)
print(f"Input: Discharge Summary (Note N002)\n")

pipeline_result = run_full_pipeline(
    raw_note=CLINICAL_NOTES[1]["text"],
    note_id="N002",
)

print(f"\nPipeline status: {pipeline_result['status']}")
print(f"Steps completed: {len(pipeline_result['steps'])}")

In [ ]:
# --- Display pipeline results as a structured report ---

print("PIPELINE OUTPUT REPORT")
print("=" * 60)

# Entities
ents = pipeline_result["steps"]["entities"]
print("\n--- MEDICATIONS ---")
for med in ents.get("medications", []):
    status_icon = {"active": "  ", "new": "++", "hold": "!!", "discontinued": "--"}.get(
        med.get("status", "active"), "  "
    )
    print(f"  [{status_icon}] {med['name']} {med['dose']} {med.get('route', '')} "
          f"{med.get('frequency', '')}")

print("\n--- DIAGNOSES ---")
for dx in ents.get("diagnoses", []):
    print(f"  {dx['description']} ({dx.get('icd10_code', 'N/A')}) [{dx.get('status', '')}]")

print("\n--- ABNORMAL LABS ---")
for lab in pipeline_result["steps"]["labs"]:
    if lab["flag"] != "normal":
        print(f"  {lab['test_name']}: {lab['value']} {lab['unit']} [{lab['flag'].upper()}]")

print("\n--- DRUG INTERACTIONS ---")
for ix in pipeline_result["steps"]["interactions"]:
    print(f"  [{ix['severity']}] {ix['drug_a']} + {ix['drug_b']}: {ix['effect'][:80]}...")

print("\n--- SOAP ASSESSMENT ---")
soap = pipeline_result["steps"]["soap_note"]
primary = soap.get("assessment", {}).get("primary_diagnosis", {})
print(f"  Primary: {primary.get('description', 'N/A')} ({primary.get('icd10', 'N/A')})")
for sec in soap.get("assessment", {}).get("secondary_diagnoses", []):
    print(f"  Secondary: {sec.get('description', 'N/A')} ({sec.get('icd10', 'N/A')})")

---
## Section 9: Results Summary & Evaluation

Let's summarize what the pipeline accomplished across all notes and evaluate entity extraction quality by comparing regex and LLM outputs.

In [ ]:
# --- Build a summary table of extracted entities across all notes ---

summary_rows = []
for note_id, entities in all_entities.items():
    note_info = next(n for n in CLINICAL_NOTES if n["note_id"] == note_id)
    summary_rows.append({
        "Note ID": note_id,
        "Type": note_info["encounter_type"],
        "Date": note_info["date"],
        "Medications": len(entities.get("medications", [])),
        "Diagnoses": len(entities.get("diagnoses", [])),
        "Allergies": len(entities.get("allergies", [])),
        "Procedures": len(entities.get("procedures", [])),
    })

df_summary = pd.DataFrame(summary_rows)
print("Entity Extraction Summary Across All Notes")
print("=" * 70)
print(df_summary.to_string(index=False))
print(f"\nTotals:")
print(f"  Medications: {df_summary['Medications'].sum()}")
print(f"  Diagnoses:   {df_summary['Diagnoses'].sum()}")
print(f"  Allergies:   {df_summary['Allergies'].sum()}")
print(f"  Procedures:  {df_summary['Procedures'].sum()}")

In [ ]:
# --- Track medication changes across encounters ---

print("MEDICATION RECONCILIATION ACROSS ENCOUNTERS")
print("=" * 65)

for note_id in ["N001", "N003", "N002", "N004"]:
    entities = all_entities[note_id]
    note_info = next(n for n in CLINICAL_NOTES if n["note_id"] == note_id)
    print(f"\n[{note_id}] {note_info['date']} - {note_info['encounter_type']}")
    print("-" * 50)
    for med in entities.get("medications", []):
        status = med.get("status", "active").upper()
        status_icon = {"ACTIVE": "  ", "NEW": "+>", "HOLD": "!!", "DISCONTINUED": "X>"}.get(
            status, "  "
        )
        print(f"  {status_icon} [{status:>12}] {med['name']} {med['dose']} "
              f"{med.get('frequency', '')}")

---
## Summary

In this notebook we built a complete **Medical Record Summarization Pipeline**:

1. **De-identification** -- Removed PHI from clinical notes using HIPAA Safe Harbor regex patterns
2. **Entity Extraction** -- Hybrid regex + LLM approach for medications, diagnoses, allergies, vitals, and labs
3. **SOAP Note Generation** -- Converted free-text notes into structured SOAP format using JSON mode
4. **Drug Interaction Checking** -- Cross-referenced medication pairs against a curated interaction database
5. **Patient Timeline** -- Generated chronological event timeline across multiple encounters
6. **Handoff Summary** -- Produced I-PASS formatted shift-change summaries
7. **Full Pipeline** -- Wired everything end-to-end: raw note in, structured report out

### Key Takeaways
- **De-identify FIRST** -- Never send raw PHI to a cloud LLM API without de-identification or a BAA
- **Hybrid extraction beats either approach alone** -- Regex for precision, LLM for recall
- **Structured output (JSON mode) is essential** -- Clinical data must be machine-readable for downstream systems
- **Confidence scoring enables clinician triage** -- High-confidence auto-populate, low-confidence flag for review
- **Clinician-in-the-loop is mandatory** -- LLM outputs are suggestions, never autonomous clinical decisions
- **Drug interaction checking is a safety-critical layer** -- Tiered severity prevents alert fatigue

### Production Requirements (NOT covered in this demo)
- HIPAA-compliant infrastructure (BAAs, encryption, access controls)
- FDA regulatory review for clinical decision support
- EHR integration via FHIR standards
- Prospective clinical validation under IRB oversight
- Model versioning and regression testing

**Next use case**: `04-earnings-call-analyzer.html` -- Financial document analysis with LLMs